# Zero-shot Baseline: Qwen2-VL-2B on VQA-RAD

**STAT GR5293 GenAI Course Project — Member 1, Weeks 1–4**

This notebook is the **main entry point** for the zero-shot baseline. It produces the headline numbers (and 95% bootstrap CIs) that LoRA, QLoRA, and DoRA experiments will be compared against.

## What this notebook does

1. Verifies a GPU is available
2. Installs pinned dependencies
3. (Optional) mounts Google Drive to cache the model + dataset
4. Sets a global random seed for reproducibility
5. Runs a 5-example smoke test
6. **Runs the full 451-example evaluation**
7. Computes 95% bootstrap confidence intervals
8. Saves all artifacts to `results/baseline/`
9. Visualizes the results
10. Produces the final hand-off table for Members 2 and 3

## Expected runtime

On a free-tier Colab T4 GPU: **~25 minutes total**.

* Dependencies: ~2 min
* Model download (Qwen2-VL-2B, ~4 GB): ~3 min
* Smoke test: ~2 min
* Full 451-example evaluation: ~15–20 min
* Bootstrap CIs: ~10 sec

## Before you start

**Runtime → Change runtime type → T4 GPU → Save**. The first cell asserts a GPU is available and refuses to continue on CPU.

## 1. Environment check

In [ ]:
# --- GPU check ---
import torch

assert torch.cuda.is_available(), (
    'No GPU detected. Inference would take hours on CPU.\n'
    'Fix: Runtime → Change runtime type → T4 GPU → Save, then re-run this cell.'
)

print(f'GPU         : {torch.cuda.get_device_name(0)}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA        : {torch.version.cuda}')

In [ ]:
# --- Install dependencies ---
# Colab pre-installs torch with the right CUDA version. We DO NOT touch torch.
# The output here will be noisy; ignore everything except errors at the end.
%pip install -q --upgrade \
    "transformers>=4.45.0,<4.50.0" \
    "accelerate>=0.34.0" \
    "datasets>=3.0.0" \
    "qwen-vl-utils>=0.0.8" \
    "rouge-score>=0.1.2" \
    "nltk>=3.9" \
    "sacrebleu>=2.4.0"

In [ ]:
# --- (Optional) Mount Google Drive for caching ---
# Set USE_DRIVE = True to cache the 4 GB Qwen2-VL-2B download on your Drive,
# so it survives Colab session restarts. Set USE_DRIVE = False to skip.
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    HF_CACHE = '/content/drive/MyDrive/peft-vqa-cache'
    os.makedirs(HF_CACHE, exist_ok=True)
    print(f'HF cache: {HF_CACHE}')
else:
    HF_CACHE = None
    print('Not using Drive cache. Model will re-download on each session.')

## 2. Project source code

Make sure the project is at `/content/peft-medical-vqa`. If you uploaded a zip and unzipped it elsewhere, edit `PROJECT_DIR` accordingly.

In [ ]:
PROJECT_DIR = '/content/peft-medical-vqa'

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f'{PROJECT_DIR} not found.\n'
        'Either:\n'
        '  1. !git clone <repo_url> /content/peft-medical-vqa, or\n'
        '  2. Upload peft-medical-vqa.zip via the file panel and run\n'
        '     !unzip -q peft-medical-vqa.zip -d /content/'
    )

%cd {PROJECT_DIR}
import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# --- Set global seed for full reproducibility ---
from src.utils.seed import set_global_seed
set_global_seed(42)
print('Seeded all RNGs with 42.')

## 3. Smoke test (5 examples)

If this works, the full run will work. If it fails, the full run would have wasted 20 minutes.

In [ ]:
from src.evaluation.evaluate_baseline import run_baseline

smoke_metrics = run_baseline(
    split='test',
    max_examples=5,
    output_dir='results/baseline_smoke',
    cache_dir=HF_CACHE,
)
print('\n✅ Smoke test passed. Proceed to the full run below.')

## 4. Full baseline evaluation

This is the actual evaluation on all 451 test examples. **15–20 minutes on T4.** Don't close the browser tab while it runs.

In [ ]:
metrics = run_baseline(
    model_id='Qwen/Qwen2-VL-2B-Instruct',
    split='test',
    max_examples=None,                # full 451 examples
    output_dir='results/baseline',
    cache_dir=HF_CACHE,
    bootstrap_resamples=10000,
    seed=42,
)

## 5. Inspect the results

In [ ]:
import json
import pandas as pd

# Load aggregated metrics + 95% CIs
with open('results/baseline/metrics.json') as f:
    m = json.load(f)
print(json.dumps(m, indent=2))

In [ ]:
# Load per-example predictions and inspect a sample
preds_df = pd.read_json('results/baseline/predictions.jsonl', lines=True)
print(f'Total predictions: {len(preds_df)}')
print(f'Closed-ended:     {(preds_df.qtype == "closed").sum()}')
print(f'Open-ended:       {(preds_df.qtype == "open").sum()}')
preds_df.head(10)

In [ ]:
# Error analysis: which closed-ended questions did the model get wrong?
from src.evaluation.metrics import normalize_text

closed = preds_df[preds_df.qtype == 'closed'].copy()
closed['correct'] = closed.apply(
    lambda r: normalize_text(r.prediction) == normalize_text(r.reference), axis=1
)
wrong = closed[~closed.correct]
print(f'Closed-ended errors: {len(wrong)}/{len(closed)} '
      f'({100*len(wrong)/len(closed):.1f}%)')
wrong.head(10)[['question', 'reference', 'prediction']]

In [ ]:
# Bias analysis: is the model systematically guessing 'yes' or 'no'?
# A balanced model on a balanced test set should have similar 'said yes' rates
# across yes-truth and no-truth examples.
yn = closed.copy()
yn['ref_norm']  = yn.reference.map(normalize_text)
yn['pred_norm'] = yn.prediction.map(normalize_text)

bias_table = pd.crosstab(yn.ref_norm, yn.pred_norm.str[:3], margins=True,
                          rownames=['Reference'], colnames=['Predicted (first 3 chars)'])
print('Confusion matrix on closed-ended questions:')
bias_table

## 6. Visualize the results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: bar chart with 95% CI error bars on EM
labels = ['Closed EM', 'Open BLEU-1', 'Open ROUGE-L', 'Open F1', 'Overall EM']
values = [m['closed']['exact_match'], m['open']['bleu1'],
          m['open']['rougeL'], m['open']['f1'], m['overall']['exact_match']]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']

# Error bars (95% CI) where available
yerr_lower = [0]*5
yerr_upper = [0]*5
if 'ci95' in m['closed']:
    yerr_lower[0] = values[0] - m['closed']['ci95']['lower']
    yerr_upper[0] = m['closed']['ci95']['upper'] - values[0]
if 'ci95_f1' in m['open']:
    yerr_lower[3] = values[3] - m['open']['ci95_f1']['lower']
    yerr_upper[3] = m['open']['ci95_f1']['upper'] - values[3]
if 'ci95' in m['overall']:
    yerr_lower[4] = values[4] - m['overall']['ci95']['lower']
    yerr_upper[4] = m['overall']['ci95']['upper'] - values[4]

axes[0].bar(labels, values, color=colors,
             yerr=[yerr_lower, yerr_upper], capsize=4, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].set_title('Zero-shot baseline scores (error bars: 95% bootstrap CI)')
for tick in axes[0].get_xticklabels():
    tick.set_rotation(20); tick.set_ha('right')

# Right: split composition
type_counts = preds_df.qtype.value_counts()
axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
             colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title(f'VQA-RAD test split composition (n={len(preds_df)})')

plt.tight_layout()
fig.savefig('results/baseline/baseline_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved figure → results/baseline/baseline_summary.png')

## 7. Hand-off table for downstream PEFT experiments

Members 2 (LoRA / QLoRA) and 3 (DoRA) compare against this baseline. The numbers below are the **single source of truth** — copy them directly into the final report's results table.

In [ ]:
def fmt_ci(point, ci_dict):
    """Format a point estimate with its 95% CI."""
    if ci_dict is None:
        return f'{point:.4f}'
    return f'{point:.4f} [{ci_dict["lower"]:.4f}, {ci_dict["upper"]:.4f}]'

summary = pd.DataFrame([
    {
        'Method': 'Baseline (Qwen2-VL-2B, zero-shot)',
        'Closed EM': fmt_ci(m['closed']['exact_match'], m['closed'].get('ci95')),
        'Open BLEU-1': f"{m['open']['bleu1']:.4f}",
        'Open ROUGE-L': f"{m['open']['rougeL']:.4f}",
        'Open F1': fmt_ci(m['open']['f1'], m['open'].get('ci95_f1')),
        'Overall EM': fmt_ci(m['overall']['exact_match'], m['overall'].get('ci95')),
        'Trainable Params': '0',
        'Peak GPU (GB)': f"{m['meta']['peak_gpu_memory_gb']:.2f}",
        'Inference (s/ex)': f"{m['meta']['inference_seconds'] / max(m['meta']['n_examples'], 1):.2f}",
    },
])
summary

## 8. Final summary

If everything ran without error, you should now have:

* `results/baseline/metrics.json` — aggregated metrics with 95% CIs
* `results/baseline/predictions.jsonl` — 451 per-example predictions
* `results/baseline/per_example_scores.json` — per-example scores for paired statistical tests
* `results/baseline/baseline_summary.png` — visualization

These four files are the deliverables for the baseline phase. They feed directly into:

1. The Methodology and Results sections of the final report
2. The paired statistical tests (McNemar / paired bootstrap) that Members 2 and 3 will run to test "is LoRA significantly better than baseline?"
3. The headline comparison table in the project presentation